In [1]:
import torch

from PIL import Image
from transformers import AutoProcessor, CLIPModel

MODEL_NAME = "openai/clip-vit-base-patch32"

# 실제 신경망 역할
# from_pretrained는 이미 학습된 모델/전처리 설정을 가져오는 메서드
model = CLIPModel.from_pretrained(MODEL_NAME)
# 이미지/텍스트를 CLIP에 넣을 수 있는 형태로 전처리
# 이미 학습된 Image Encoder, Text Encoder 을 사용함
# PIL 이미지 -> resize -> normalize -> Tensor
# 문자열 -> tokenize -> input_ids -> Tensor
processor = AutoProcessor.from_pretrained(MODEL_NAME)

# 각각의 이미지 및 문장을 CLIP에 넣어주게 되면 image/text embedding 이 되게 됩니다.

print(type(model))
print(type(processor))

c:\LANG_CHAIN_2026\2026-05-19_KDT_lang_chain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 18165.26it/s]


<class 'transformers.models.clip.modeling_clip.CLIPModel'>
<class 'transformers.models.clip.processing_clip.CLIPProcessor'>


In [2]:
from pathlib import Path
Path("images/cchamppang.png").exists()

True

In [15]:
breakpoint()

image = Image.open(Path("images/cchamppang.png"))

image = image.convert("RGB")

inputs = processor(
  images=image,
  return_tensors="pt"
)
print(inputs["pixel_values"].shape)
print(inputs)

torch.Size([1, 3, 224, 224])
{'pixel_values': tensor([[[[1.9157, 1.9157, 1.9157,  ..., 1.9157, 1.9157, 1.9157],
          [1.9157, 1.9157, 1.9157,  ..., 1.9157, 1.9157, 1.9157],
          [1.9157, 1.9157, 1.9157,  ..., 1.9157, 1.9303, 1.9303],
          ...,
          [1.9157, 1.9157, 1.9157,  ..., 1.9157, 1.9157, 1.9303],
          [1.9157, 1.9157, 1.9157,  ..., 1.9157, 1.9157, 1.9157],
          [1.9157, 1.9157, 1.9157,  ..., 1.9157, 1.9157, 1.9157]],

         [[2.0599, 2.0599, 2.0599,  ..., 2.0599, 2.0599, 2.0599],
          [2.0599, 2.0599, 2.0599,  ..., 2.0599, 2.0599, 2.0599],
          [2.0599, 2.0599, 2.0599,  ..., 2.0599, 2.0599, 2.0599],
          ...,
          [2.0599, 2.0599, 2.0599,  ..., 2.0599, 2.0599, 2.0749],
          [2.0599, 2.0599, 2.0599,  ..., 2.0599, 2.0599, 2.0599],
          [2.0599, 2.0599, 2.0599,  ..., 2.0599, 2.0599, 2.0599]],

         [[2.1317, 2.1317, 2.1317,  ..., 2.1317, 2.1317, 2.1459],
          [2.1317, 2.1317, 2.1317,  ..., 2.1317, 2.1317, 2.131

In [27]:
with torch.no_grad():
  # 모델 추론: 이미지 -> 벡터
  output = model.get_image_features(
    pixel_values=inputs["pixel_values"]
  )
image_features = output.pooler_output
hidden_state = output.last_hidden_state

# Hugging Face의 CLIP 구현에는 get_image_features가 출력 객체를 반환하고
# 실제 이미지 임베딩은 그 안의 pooler_output에 존재함
print(hidden_state.shape)
print(image_features.shape)
print(image_features)

torch.Size([1, 50, 768])
torch.Size([1, 512])
tensor([[-5.3123e-03,  6.4366e-02,  2.2283e-01, -9.2436e-03,  3.9275e-01,
         -7.3967e-01, -3.0512e-01,  1.6983e-01,  1.8343e-01, -2.7675e-01,
          1.8894e-01, -8.9344e-02,  1.3473e+00,  1.2617e-01,  5.2795e-01,
         -1.3225e-01,  2.0388e-01, -4.4995e-02, -5.8420e-02,  2.6017e-01,
         -2.3787e-01,  3.4642e-01, -4.3586e-01, -1.5265e-01, -2.0086e-01,
          4.9386e-01, -4.7573e-01,  7.0535e-01,  1.4024e-01,  3.5122e-01,
          3.3361e-02,  8.0742e-03,  1.0590e-01,  1.0539e+00,  2.6119e-01,
          2.5047e-01, -2.8579e-01,  2.2777e-01,  4.9199e-01, -1.1601e+00,
         -1.8739e-01, -1.5655e-01, -2.4976e-02,  2.8328e-01, -1.2027e-01,
          1.7717e+00, -3.1691e-01,  2.8188e-01,  4.1053e-01, -2.8790e-02,
          8.5738e-02,  9.1758e-02,  1.2631e-01,  1.3558e-01, -6.6162e-02,
         -2.1972e-01,  7.7159e-01,  2.5869e-01,  3.8645e-01,  1.2127e-01,
          1.9651e+00, -4.1428e-01,  9.0407e-02, -1.1305e-01, -6.13

In [14]:
processor(
  text=["a dog", "a cat"],
  images=image,
  return_tensors="pt"
)["input_ids"]
# processor(
#   text=["a dog", "a cat"],
#   images=image,
#   return_tensors="pt"
# )["pixel_values"].shape

tensor([[49406,   320,  1929, 49407],
        [49406,   320,  2368, 49407]])

In [28]:
image_features.shape

torch.Size([1, 512])

In [30]:
import torch.nn.functional as F

with torch.no_grad():
  outputs = model.get_image_features(
    pixel_values=inputs["pixel_values"]
  )

image_features = outputs.pooler_output

image_features = F.normalize(image_features, p=2, dim=-1)
print(image_features.shape)

torch.Size([1, 512])


In [31]:
texts = [
  "a dog",
  "a cat",
  "a car"
]

# 텍스트 텐서화
text_inputs = processor(
  text=texts,
  return_tensors="pt",
  padding=True
)

print(text_inputs)

{'input_ids': tensor([[49406,   320,  1929, 49407],
        [49406,   320,  2368, 49407],
        [49406,   320,  1615, 49407]]), 'attention_mask': tensor([[1, 1, 1, 1],
        [1, 1, 1, 1],
        [1, 1, 1, 1]])}


In [37]:
with torch.no_grad():
  text_outputs = model.get_text_features(
    input_ids=text_inputs["input_ids"],
    attention_mask=text_inputs["attention_mask"]
  )

# 텍스트 임베딩 확인
text_outputs.pooler_output.shape

torch.Size([3, 512])

In [38]:
# 정규화
text_features = text_outputs.pooler_output

text_features = F.normalize(
  text_features,
  p=2,
  dim=-1
)
print(text_features.shape)

torch.Size([3, 512])


In [45]:
print((text_features**2).sum(dim=1))

tensor([1.0000, 1.0000, 1.0000])


In [47]:
image_features.shape

torch.Size([1, 512])

In [ ]:
# 이미 모두 길이 1짜리 벡터이기 때문에 이 값은 사실상 `cosine similarity`
similarities = image_features @ text_features.T

print(similarities)
print(similarities.shape)

tensor([[0.2293, 0.2345, 0.2143]])
torch.Size([1, 3])


In [52]:
text = ["a guy"]

text_input = processor(
  text=text,
  return_tensors="pt",
  padding=True
)
with torch.no_grad():
  text_output = model.get_text_features(
    input_ids=text_input["input_ids"],
    attention_mask=text_input["attention_mask"]
  )
# text_output.pooler_output.shape

text_feature = F.normalize(text_output.pooler_output, p=2, dim=-1)
text_feature.shape

torch.Size([1, 512])

In [53]:
similarity = image_features @ text_feature.T
similarity

tensor([[0.2439]])

In [60]:
similarities.shape, similarity.shape

(torch.Size([1, 3]), torch.Size([1, 1]))

In [68]:
similarities = torch.hstack((similarities, similarity))

In [66]:
texts.extend(text)
texts

['a dog', 'a cat', 'a car', 'a guy']

In [69]:
best_idx = similarities.argmax(dim=-1).item()
print("가장 유사한 문장:", texts[best_idx])
print("유사도:", similarities[0, best_idx].item())

가장 유사한 문장: a guy
유사도: 0.2438749372959137
